## Parse Structured XML Generated by Grobid

In [ ]:
import json
import logging
import Levenshtein
import os
import requests

from bs4 import BeautifulSoup

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [ ]:
TEI_XML_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/tei/'
TEI_XML_CROSSREF_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/tei_crossref_citations/'
GROUPED_REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/grouped_refs_full/'
REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/refs_grobid/'

LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD = 5

In [ ]:
def extract_grouped_refs(soup):
    """
    Extract references grouped by occurrence in the same paragraph of the paper.
    """
    text = soup.find('text')
    grouped_refs = []
    for child_tag in text.body.children:
        # Skip empty lines and non-text tags (e.g., notes)
        if child_tag.name != 'div':
            continue

        # Extract title if present
        title = child_tag.head.text if child_tag.head else ''

        # Group all references from a single div
        ref_tags = child_tag.find_all('ref', attrs={"type": "bibr"})
        refs = [(ref_tag.get('target', None), ref_tag.text)
                for ref_tag in ref_tags]
        grouped_refs.append({'title': title, 'references': refs})
    return grouped_refs

In [ ]:
def extract_refs(soup):
    """
    Extract list of references for the paper.
    """
    bibliography = soup.find('listBibl')
    refs = {}
    for bibl_struct in bibliography.children:
        # Skip empty lines
        if bibl_struct.name != 'biblStruct':
            continue

        ref_id = bibl_struct['xml:id']
        ref_text = ''
        if bibl_struct.analytic and bibl_struct.analytic.title:
            ref_text = bibl_struct.analytic.title.text
        refs[ref_id] = ref_text
    return refs

In [ ]:
def parse_xml(filename):
    with open(filename, 'r') as xml_file:
        soup = BeautifulSoup(xml_file, 'xml')
    
    grouped_refs = extract_grouped_refs(soup)
    refs = extract_refs(soup)
    return grouped_refs, refs

In [ ]:
def export_to_json(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)

In [ ]:
def process_xml(filename):
    pmid = filename.split('.')[0]
    xml_raw = os.path.join(TEI_XML_FOLDER, filename)
    xml_crossref = os.path.join(TEI_XML_CROSSREF_FOLDER, filename)

    grouped_refs_raw, refs_raw = parse_xml(xml_raw)
    grouped_refs_crossref, refs_crossref = parse_xml(xml_crossref)
    
    if grouped_refs_raw != grouped_refs_crossref:
        print('Grouped refs raw vs crossref differ')
        
    refs_merged = {}
    for ref_id, ref_raw in refs_raw.items():
        ref_fixed = refs_crossref[ref_id]
        distance = Levenshtein.distance(ref_raw.lower(), ref_fixed.lower())
        
        selected = False
        reason = ''
        if distance > LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD and ref_raw:
            selected = True
            reason = ref_raw
        merged_ref = {
            'title': ref_fixed,
            'selected': selected,
            'reason': reason
        }

        refs_merged[ref_id] = merged_ref
    
    grouped_refs_filename = os.path.join(GROUPED_REFS_FOLDER, f'{pmid}.json')
    export_to_json(grouped_refs_raw, grouped_refs_filename)
        
    refs_filename = os.path.join(REFS_FOLDER, f'{pmid}.json')
    # Skip merging
    export_to_json(refs_raw, refs_filename)

In [ ]:
for folder in [GROUPED_REFS_FOLDER, REFS_FOLDER]:
    if not os.path.exists(folder):
        os.mkdir(folder)
for filename in os.listdir(TEI_XML_FOLDER):
    if filename.endswith('.xml'):
        logging.info(f'Processing {filename}')
        process_xml(filename)

## Obtaining the Clustering with PMIDs

In [13]:
import gzip
import json
import logging
import os

from pysrc.papers.analyzer import KeyPaperAnalyzer
from pysrc.papers.pubtrends_config import PubtrendsConfig
from pysrc.papers.db.loaders import Loaders

In [14]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [2]:
GROUPED_REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/grouped_refs_validated/'
REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/refs_grobid/'
PUBTRENDS_EXPORT_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/pubtrends_export/'
CLUSTERING_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/clustering/'

In [3]:
PUBTRENDS_CONFIG = PubtrendsConfig(test=False)


def reload_exported_analyzer(path_to_archive, source='Pubmed'):
    """
    Load analysis data from json.gz archive generated by PubTrends.
    """
    with gzip.open(path_to_archive, 'rt', encoding='UTF-8') as zipfile:
        data = json.load(zipfile)

    loader, url_prefix = Loaders.get_loader_and_url_prefix(source, PUBTRENDS_CONFIG)
    analyzer = KeyPaperAnalyzer(loader, PUBTRENDS_CONFIG)
    analyzer.init(data)
    
    return analyzer

In [4]:
def reference_index(analyzer, review_pmid):
    reference_pmids = list(analyzer.citations_graph.successors(review_pmid))
    pubmed_data = analyzer.df[analyzer.df['id'].isin(reference_pmids)]
    pubmed_ref_search_index = {v.lower(): k for k, v in zip(pubmed_data['id'], pubmed_data['title'])}
    return pubmed_ref_search_index

In [5]:
def grobid2pmid(refs, reference_index):
    mapping = {}

    for key, ref in refs.items():
        if ref.lower() in reference_index:
            pmid = reference_index[ref.lower()]
            mapping[key] = int(pmid)
            
    return mapping

In [6]:
def export_to_json(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)

In [7]:
def pmid_clustering(grouped_refs, pmid_mapping, prefix=''):
    clustering = []
    for el in grouped_refs:
        if isinstance(el, dict):
            new_element = {
                'title': el['title'],
                'references': pmid_clustering(el['references'], pmid_mapping, prefix='> ')
            }
        else:
            new_element = None
            if el[0]:  # Some references do not have IDs due to parsing error
                grobid_id = el[0][1:]  # '#b0' -> 'b0'
                if grobid_id in pmid_mapping:
                    new_element = pmid_mapping[grobid_id]
    
        if new_element:
            clustering.append(new_element)
            
    return clustering

In [9]:
def obtain_clustering(file):
    pmid, ext = os.path.splitext(file)
    
    logging.info(f'{pmid}: loading file with grouped references')
    filename = os.path.join(GROUPED_REFS_FOLDER, file)
    with open(filename, 'r') as f:
        data = json.load(f)

    logging.info(f'{pmid}: loading file with PubTrends analyzer')
    analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
    analyzer = reload_exported_analyzer(analysis_file)
    
    logging.info(f'{pmid}: building reference index for matching titles and PMIDs')
    ref_index = reference_index(analyzer, pmid)
    
    logging.info(f'{pmid}: loading file with references')
    refs_file = os.path.join(REFS_FOLDER, file)
    with open(refs_file, 'r') as f:
        refs = json.load(f)
        
    logging.info(f'{pmid}: building mapping between Grobid reference IDs and PMIDs')
    mapping = grobid2pmid(refs, ref_index)
    print(f'{pmid}: {len(mapping)} / {analyzer.citations_graph.out_degree(pmid)} references mapped')
    
    logging.info(f'{pmid}: building clustering with PMIDs instead of Grobid IDs')
    clustering = pmid_clustering(data, mapping)
    clustering_file = os.path.join(CLUSTERING_FOLDER, file)
    
    logging.info(f'{pmid}: exporting clustering')
    export_to_json(clustering, clustering_file)
    
    logging.info(f'{pmid}: done\n')
    return len(mapping), analyzer.citations_graph.out_degree(pmid)

In [15]:
if not os.path.exists(CLUSTERING_FOLDER):
    os.mkdir(CLUSTERING_FOLDER)

refs_mapped = 0
refs_total = 0
for file in os.listdir(GROUPED_REFS_FOLDER):
    new_refs_mapped, new_refs_total = obtain_clustering(file)
    refs_mapped += new_refs_mapped
    refs_total += new_refs_total

2021-02-03 03:05:25,390 INFO: 30108335: loading file with grouped references
2021-02-03 03:05:25,439 INFO: 30108335: loading file with PubTrends analyzer
2021-02-03 03:05:26,901 INFO: 30108335: building reference index for matching titles and PMIDs
2021-02-03 03:05:26,905 INFO: 30108335: loading file with references
2021-02-03 03:05:26,925 INFO: 30108335: building mapping between Grobid reference IDs and PMIDs
2021-02-03 03:05:26,927 INFO: 30108335: building clustering with PMIDs instead of Grobid IDs
2021-02-03 03:05:26,928 INFO: 30108335: exporting clustering
2021-02-03 03:05:26,933 INFO: 30108335: done

2021-02-03 03:05:27,022 INFO: 28792006: loading file with grouped references
2021-02-03 03:05:27,067 INFO: 28792006: loading file with PubTrends analyzer


30108335: 231 / 277 references mapped


2021-02-03 03:05:27,874 INFO: 28792006: building reference index for matching titles and PMIDs
2021-02-03 03:05:27,878 INFO: 28792006: loading file with references
2021-02-03 03:05:27,880 INFO: 28792006: building mapping between Grobid reference IDs and PMIDs
2021-02-03 03:05:27,882 INFO: 28792006: building clustering with PMIDs instead of Grobid IDs
2021-02-03 03:05:27,883 INFO: 28792006: exporting clustering
2021-02-03 03:05:27,887 INFO: 28792006: done

2021-02-03 03:05:27,930 INFO: 30390028: loading file with grouped references
2021-02-03 03:05:27,947 INFO: 30390028: loading file with PubTrends analyzer


28792006: 105 / 126 references mapped


2021-02-03 03:05:28,827 INFO: 30390028: building reference index for matching titles and PMIDs
2021-02-03 03:05:28,830 INFO: 30390028: loading file with references
2021-02-03 03:05:28,861 INFO: 30390028: building mapping between Grobid reference IDs and PMIDs
2021-02-03 03:05:28,863 INFO: 30390028: building clustering with PMIDs instead of Grobid IDs
2021-02-03 03:05:28,865 INFO: 30390028: exporting clustering
2021-02-03 03:05:28,870 INFO: 30390028: done

2021-02-03 03:05:28,918 INFO: 29213134: loading file with grouped references
2021-02-03 03:05:28,928 INFO: 29213134: loading file with PubTrends analyzer


...

26678314: 144 / 198 references mapped


2021-02-03 03:05:43,471 INFO: 27834397: building reference index for matching titles and PMIDs
2021-02-03 03:05:43,475 INFO: 27834397: loading file with references
2021-02-03 03:05:43,479 INFO: 27834397: building mapping between Grobid reference IDs and PMIDs
2021-02-03 03:05:43,481 INFO: 27834397: building clustering with PMIDs instead of Grobid IDs
2021-02-03 03:05:43,485 INFO: 27834397: exporting clustering
2021-02-03 03:05:43,489 INFO: 27834397: done

2021-02-03 03:05:43,545 INFO: 26580717: loading file with grouped references
2021-02-03 03:05:43,558 INFO: 26580717: loading file with PubTrends analyzer


27834397: 172 / 200 references mapped


2021-02-03 03:05:44,575 INFO: 26580717: building reference index for matching titles and PMIDs
2021-02-03 03:05:44,579 INFO: 26580717: loading file with references
2021-02-03 03:05:44,610 INFO: 26580717: building mapping between Grobid reference IDs and PMIDs
2021-02-03 03:05:44,612 INFO: 26580717: building clustering with PMIDs instead of Grobid IDs
2021-02-03 03:05:44,614 INFO: 26580717: exporting clustering
2021-02-03 03:05:44,617 INFO: 26580717: done



26580717: 78 / 91 references mapped


In [16]:
refs_mapped / refs_total

0.7637893424742911

## Bulk PubTrends Export

In [ ]:
import gzip
import json
import logging
import os

from pysrc.papers.analyzer import KeyPaperAnalyzer
from pysrc.papers.pubtrends_config import PubtrendsConfig, DEFAULT_ANALYZER_SETTINGS
from pysrc.papers.db.loaders import Loaders
from pysrc.papers.db.search_error import SearchError
from pysrc.papers.plot.plotter import visualize_analysis
from pysrc.papers.progress import Progress
from pysrc.papers.utils import SORT_MOST_CITED, ZOOM_OUT, PAPER_ANALYSIS

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [ ]:
TARGET_FOLDER = '/home/willenjoy/work/pubtrends/local/pubtrends_export/'

TARGET_PMIDS = [26667849, 26678314, 26688349, 26688350, 26580716, 26580717, 26656254, 26675821, 27834397, 27834398,
                27890914, 27916977, 27677859, 27677860, 27904142, 28003656, 29147025, 29170536, 28853444, 28920587,
                28792006, 28852220, 29213134, 29321682, 30578414, 30842595, 30644449, 30679807, 30108335, 30390028,
                30459365, 30467385, 31686003, 31806885, 31836872, 32005979, 31937935, 32020081, 32042144, 32699292]

SOURCE = 'Pubmed'
LIMIT = 500

In [ ]:
def export_analysis(pmid):
    logging.info(f'Started analysis for PMID {pmid}')
    
    ids = [pmid]
    query = f'Paper ID: {pmid}'
    
    # extracted from 'analyze_id_list' Celery task
    config = PubtrendsConfig(test=False)
    loader = Loaders.get_loader(SOURCE, config)
    analyzer = KeyPaperAnalyzer(loader, config)
    settings = DEFAULT_ANALYZER_SETTINGS
    try:
        # Fetch references at first
        ids = ids + analyzer.load_references(
            ids[0], limit=LIMIT
        )
        # And then expand
        ids = analyzer.expand_ids(
            ids, limit=LIMIT,
            expand_steps=settings.EXPAND_STEPS, expand_citations_sigma=settings.EXPAND_CITATIONS_SIGMA,
            expand_similarity_threshold=settings.EXPAND_SIMILARITY_THRESHOLD,
            current=1, task=None
        )

        analyzer.analyze_papers(ids, query, task=None)
    finally:
        loader.close_connection()

    dump = analyzer.dump()
    
    # export as JSON
    path = os.path.join(TARGET_FOLDER, f'{pmid}.json.gz')
    with gzip.open(path, 'w') as f:
        f.write(json.dumps(dump).encode('utf-8'))
    
    logging.info(f'Finished analysis for PMID {pmid}\n')

In [ ]:
TARGET_PMIDS = [49, 58, 59, 64]

In [ ]:
for pmid in TARGET_PMIDS:
    export_analysis(pmid)